# Shared Expenses CSV Import Prototype

This notebook prototypes the import/preprocessing layer for the assignment. It intentionally keeps the raw CSV immutable, builds normalized candidate records, and creates an anomaly report for anything that should be shown to a user before the app changes, deletes, or imports it.

The goal here is not to hide messy data. The goal is to prove that the data can be processed deliberately: detect, explain, and route each problem according to a documented policy.


In [ ]:
from pathlib import Path
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP, ROUND_FLOOR
from difflib import SequenceMatcher
import re
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 120)

CENTS = Decimal('0.01')
DEFAULT_YEAR = 2026

# Prototype-only config. In the app this must be saved with the import report,
# not hidden in code, so Priya can see exactly how USD became INR.
FX_RATES_TO_INR = {
    'INR': Decimal('1.00'),
    'USD': Decimal('83.00'),
}

EXPECTED_COLUMNS = [
    'date', 'description', 'paid_by', 'amount', 'currency',
    'split_type', 'split_with', 'split_details', 'notes'
]

PERSON_ALIASES = {
    'aisha': 'Aisha',
    'rohan': 'Rohan',
    'priya': 'Priya',
    'priya s': 'Priya',
    'meera': 'Meera',
    'dev': 'Dev',
    'sam': 'Sam',
    'kabir': 'Kabir',
    'devs friend kabir': 'Kabir',
}

FLATMATES = {'Aisha', 'Rohan', 'Priya', 'Meera', 'Sam'}
KNOWN_PEOPLE = set(PERSON_ALIASES.values())

# Policy inferred from the assignment text and CSV notes.
MEMBERSHIP_WINDOWS = {
    'Aisha': (pd.Timestamp('2026-02-01'), None),
    'Rohan': (pd.Timestamp('2026-02-01'), None),
    'Priya': (pd.Timestamp('2026-02-01'), None),
    'Meera': (pd.Timestamp('2026-02-01'), pd.Timestamp('2026-03-31')),
    'Sam': (pd.Timestamp('2026-04-08'), None),
}

REVIEW_NOTE_TERMS = [
    'wrong', 'oops', 'forgot', 'cant remember', 'not expense',
    'counted twice', 'format is a mess', 'percentages might be off',
    'i think', 'is this', 'fixing later'
]

SETTLEMENT_TERMS = [
    'paid back', 'settlement', 'deposit share', 'paid aisha his deposit'
]


## Import Policy Used In This Prototype

- Do not edit the CSV by hand.
- Preserve original row numbers so every app decision can be traced back to the uploaded file.
- Normalize safe fields such as casing, whitespace, comma-formatted amounts, and known name aliases.
- Convert currencies only through an explicit import configuration.
- Do not silently delete duplicates or questionable rows. Put them in the review queue.
- Treat settlement/payment-like rows separately from expenses.
- Hold rows for review when the date, payer, amount, split math, membership, or duplicate status is ambiguous.


In [ ]:
def find_csv_path() -> Path:
    candidates = [
        Path.cwd() / 'expense_data' / 'Expenses Export.csv',
        Path.cwd() / 'backend' / 'expense_data' / 'Expenses Export.csv',
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError('Could not find expense_data/Expenses Export.csv from the current working directory.')


CSV_PATH = find_csv_path()
raw_expenses = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
raw_expenses.insert(0, 'raw_row_number', raw_expenses.index + 2)

missing_columns = [column for column in EXPECTED_COLUMNS if column not in raw_expenses.columns]
unexpected_columns = [column for column in raw_expenses.columns if column not in EXPECTED_COLUMNS + ['raw_row_number']]

print(f'Loaded {len(raw_expenses)} raw rows from: {CSV_PATH}')
print('Missing columns:', missing_columns or 'None')
print('Unexpected columns:', unexpected_columns or 'None')
display(raw_expenses.head(8))


In [ ]:
def clean_text(value) -> str:
    if value is None or pd.isna(value):
        return ''
    return str(value).strip()


def compact_text(value, remove_stop_words=False) -> str:
    text = clean_text(value).lower().replace("'", '')
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    if remove_stop_words:
        text = re.sub(r'\b(at|the|a|an)\b', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def normalize_person(value) -> str:
    raw = clean_text(value)
    key = compact_text(raw)
    return PERSON_ALIASES.get(key, raw)


def split_people(value) -> list[str]:
    people = []
    for token in clean_text(value).split(';'):
        token = clean_text(token)
        if token:
            people.append(normalize_person(token))
    return people


def parse_amount(value) -> dict:
    raw = clean_text(value)
    normalized = raw.replace(',', '')
    result = {
        'amount_original': None,
        'amount_rounded': None,
        'amount_has_comma': ',' in raw,
        'amount_decimal_places': None,
    }
    if not normalized:
        return result
    try:
        amount = Decimal(normalized)
    except InvalidOperation:
        return result
    result['amount_original'] = amount
    result['amount_rounded'] = amount.quantize(CENTS, rounding=ROUND_HALF_UP)
    result['amount_decimal_places'] = max(0, -amount.as_tuple().exponent)
    return result


def parse_date(value):
    raw = clean_text(value)
    if re.fullmatch(r'\d{2}-\d{2}-\d{4}', raw):
        parsed = pd.to_datetime(raw, format='%d-%m-%Y', errors='coerce')
        return parsed, None if not pd.isna(parsed) else 'unparseable_standard_date'

    month_short = re.fullmatch(r'([A-Za-z]{3})-(\d{1,2})', raw)
    if month_short:
        month_name, day = month_short.groups()
        parsed = pd.to_datetime(f'{day}-{month_name}-{DEFAULT_YEAR}', format='%d-%b-%Y', errors='coerce')
        return parsed, 'non_standard_date_inferred_year' if not pd.isna(parsed) else 'unparseable_date'

    parsed = pd.to_datetime(raw, dayfirst=True, errors='coerce')
    return parsed, 'non_standard_date' if not pd.isna(parsed) else 'unparseable_date'


def parse_split_details(value) -> list[dict]:
    details = []
    raw = clean_text(value)
    if not raw:
        return details

    for token in raw.split(';'):
        token = clean_text(token)
        if not token:
            continue
        match = re.match(r'^(?P<person>.+?)\s+(?P<value>-?\d+(?:\.\d+)?)\s*(?P<percent>%?)$', token)
        if not match:
            details.append({'raw': token, 'person': '', 'value': None, 'unit': 'unparsed'})
            continue
        details.append({
            'raw': token,
            'person': normalize_person(match.group('person')),
            'value': Decimal(match.group('value')),
            'unit': 'percentage' if match.group('percent') else 'number',
        })
    return details


def money(value):
    if value is None or pd.isna(value):
        return None
    return Decimal(value).quantize(CENTS, rounding=ROUND_HALF_UP)


def allocate_weighted_amount(total, weights):
    weights = [Decimal(str(weight)) for weight in weights]
    if total is None or not weights or sum(weights) <= 0:
        return []

    sign = -1 if total < 0 else 1
    total_cents = int((abs(total) * 100).to_integral_value(rounding=ROUND_HALF_UP))
    weight_total = sum(weights)
    exact_cents = [Decimal(total_cents) * weight / weight_total for weight in weights]
    base_cents = [int(value.to_integral_value(rounding=ROUND_FLOOR)) for value in exact_cents]
    remainder = total_cents - sum(base_cents)

    ranked_indexes = sorted(
        range(len(weights)),
        key=lambda index: (exact_cents[index] - Decimal(base_cents[index]), -index),
        reverse=True,
    )
    for index in ranked_indexes[:remainder]:
        base_cents[index] += 1

    return [(Decimal(sign * cents) / Decimal(100)).quantize(CENTS) for cents in base_cents]


In [ ]:
normalized_records = []

for _, row in raw_expenses.iterrows():
    amount_info = parse_amount(row['amount'])
    parsed_date, date_issue = parse_date(row['date'])
    currency = clean_text(row['currency']).upper()
    fx_rate = FX_RATES_TO_INR.get(currency)
    amount_inr = None
    if amount_info['amount_rounded'] is not None and fx_rate is not None:
        amount_inr = (amount_info['amount_rounded'] * fx_rate).quantize(CENTS, rounding=ROUND_HALF_UP)

    payer_raw = clean_text(row['paid_by'])
    payer_normalized = normalize_person(payer_raw)
    participants_raw = [clean_text(part) for part in clean_text(row['split_with']).split(';') if clean_text(part)]
    participants = [normalize_person(part) for part in participants_raw]
    split_details = parse_split_details(row['split_details'])

    description = clean_text(row['description'])
    notes = clean_text(row['notes'])
    description_key = compact_text(description, remove_stop_words=True)
    review_text = compact_text(f'{description} {notes}')
    is_settlement_candidate = any(term in review_text for term in SETTLEMENT_TERMS)

    normalized_records.append({
        'raw_row_number': int(row['raw_row_number']),
        'date_raw': clean_text(row['date']),
        'parsed_date': parsed_date,
        'date_issue': date_issue,
        'description_raw': description,
        'description_key': description_key,
        'paid_by_raw': payer_raw,
        'paid_by_normalized': payer_normalized,
        'amount_raw': clean_text(row['amount']),
        **amount_info,
        'currency_raw': clean_text(row['currency']),
        'currency_normalized': currency,
        'fx_rate_to_inr': fx_rate,
        'amount_inr': amount_inr,
        'split_type_raw': clean_text(row['split_type']),
        'split_type': clean_text(row['split_type']).lower(),
        'split_with_raw': clean_text(row['split_with']),
        'participants_raw': participants_raw,
        'participants': participants,
        'participants_text': '; '.join(participants),
        'split_details_raw': clean_text(row['split_details']),
        'split_details_parsed': split_details,
        'split_detail_people': [detail['person'] for detail in split_details if detail['person']],
        'notes': notes,
        'is_settlement_candidate': is_settlement_candidate,
    })

normalized_expenses = pd.DataFrame(normalized_records)

display_columns = [
    'raw_row_number', 'date_raw', 'parsed_date', 'description_raw',
    'paid_by_raw', 'paid_by_normalized', 'amount_raw', 'amount_rounded',
    'currency_normalized', 'amount_inr', 'split_type', 'participants_text',
    'split_details_raw', 'notes'
]
display(normalized_expenses[display_columns].head(12))


In [ ]:
anomalies = []


def add_anomaly(row, code, severity, field, message, action, requires_user_approval=True, related_rows=''):
    anomalies.append({
        'raw_row_number': row,
        'code': code,
        'severity': severity,
        'field': field,
        'message': message,
        'action': action,
        'requires_user_approval': requires_user_approval,
        'related_rows': related_rows,
    })


supported_split_types = {'equal', 'unequal', 'percentage', 'share'}

for row in normalized_expenses.to_dict('records'):
    row_num = row['raw_row_number']
    parsed_date = row['parsed_date']
    participants = row['participants']
    participant_set = set(participants)
    split_details = row['split_details_parsed']
    detail_people = {detail['person'] for detail in split_details if detail.get('person')}

    if pd.isna(parsed_date):
        add_anomaly(row_num, 'date_unparseable', 'error', 'date', f'Could not parse date value {row["date_raw"]!r}.', 'Hold row for user correction.')
    elif row['date_issue']:
        add_anomaly(row_num, row['date_issue'], 'review', 'date', f'Date value {row["date_raw"]!r} is not the expected DD-MM-YYYY format.', 'Show inferred date and require approval before import.')

    if row['date_raw'] == '04-05-2026' or 'format is a mess' in compact_text(row['notes']):
        add_anomaly(row_num, 'ambiguous_date_note', 'review', 'date', 'Source note says the date may mean April 5 or May 4.', 'Hold row until user confirms the actual date.')

    if not row['paid_by_raw']:
        add_anomaly(row_num, 'missing_payer', 'error', 'paid_by', 'The payer is blank.', 'Hold row until the user selects who paid.')
    elif row['paid_by_raw'] != row['paid_by_normalized']:
        add_anomaly(row_num, 'payer_alias_normalized', 'warning', 'paid_by', f'Payer {row["paid_by_raw"]!r} normalized to {row["paid_by_normalized"]!r}.', 'Import using canonical person name and show the alias in the report.', False)

    participant_aliases = [
        f'{raw} -> {norm}'
        for raw, norm in zip(row['participants_raw'], participants)
        if raw != norm
    ]
    if participant_aliases:
        add_anomaly(row_num, 'participant_alias_normalized', 'warning', 'split_with', '; '.join(participant_aliases), 'Import using canonical participant names and show the aliases in the report.', False)

    unknown_people = sorted((participant_set | {row['paid_by_normalized']}) - KNOWN_PEOPLE - {''})
    if unknown_people:
        add_anomaly(row_num, 'unknown_person', 'review', 'people', f'Unknown people found: {unknown_people}.', 'Ask user whether to create guest/member records.')

    if 'Kabir' in participant_set:
        add_anomaly(row_num, 'external_guest_participant', 'review', 'split_with', 'Kabir appears as Dev\'s friend for one day.', 'Create as a guest participant only if the user approves.')

    if not row['currency_normalized']:
        add_anomaly(row_num, 'missing_currency', 'review', 'currency', 'Currency is blank.', 'Default to INR only after user approval.')
    elif row['currency_normalized'] not in FX_RATES_TO_INR:
        add_anomaly(row_num, 'unsupported_currency', 'error', 'currency', f'Currency {row["currency_normalized"]!r} has no configured FX rate.', 'Hold row until a rate is configured.')
    elif row['currency_normalized'] != 'INR':
        add_anomaly(row_num, 'foreign_currency_converted', 'warning', 'currency', f'{row["currency_normalized"]} converted to INR at {row["fx_rate_to_inr"]}.', 'Import using the explicit FX rate and include it in the report.', False)

    if row['amount_original'] is None:
        add_anomaly(row_num, 'amount_unparseable', 'error', 'amount', f'Could not parse amount {row["amount_raw"]!r}.', 'Hold row until amount is corrected.')
    else:
        if row['amount_has_comma']:
            add_anomaly(row_num, 'amount_thousands_separator', 'warning', 'amount', f'Amount {row["amount_raw"]!r} contains a comma.', 'Strip comma for numeric import and keep raw value for traceability.', False)
        if row['amount_decimal_places'] and row['amount_decimal_places'] > 2:
            add_anomaly(row_num, 'amount_precision_over_two_decimals', 'warning', 'amount', f'Amount {row["amount_raw"]!r} has more than two decimals.', 'Round half-up to two decimals for the candidate import and show the original value.', False)
        if row['amount_original'] < 0:
            add_anomaly(row_num, 'negative_amount_refund_candidate', 'review', 'amount', 'Negative amount likely represents a refund or credit, not a normal expense.', 'Classify as refund after user approval.')
        if row['amount_original'] == 0:
            add_anomaly(row_num, 'zero_amount_candidate', 'review', 'amount', 'Zero amount row does not affect balances.', 'Exclude from balances unless user confirms it is a correction record.')

    if row['is_settlement_candidate']:
        add_anomaly(row_num, 'payment_or_settlement_candidate', 'review', 'description', 'Description or notes look like a payment/settlement rather than a shared expense.', 'Route to payments/settlements instead of importing as an expense.')

    if not row['split_type']:
        add_anomaly(row_num, 'missing_split_type', 'review', 'split_type', 'Split type is blank.', 'Infer only if user confirms this is a payment/settlement or selects a split type.')
    elif row['split_type'] not in supported_split_types:
        add_anomaly(row_num, 'unsupported_split_type', 'error', 'split_type', f'Split type {row["split_type"]!r} is not supported.', 'Hold row until the importer supports this split type.')

    if row['split_details_raw'] and row['split_type'] == 'equal':
        add_anomaly(row_num, 'split_type_details_conflict', 'review', 'split_details', 'split_type is equal but split_details are present.', 'Ask user whether equal split or explicit details should win.')

    if row['split_type'] in {'unequal', 'percentage', 'share'} and not split_details:
        add_anomaly(row_num, 'missing_split_details', 'error', 'split_details', f'{row["split_type"]} split needs split_details.', 'Hold row until details are provided.')

    if detail_people and not detail_people.issubset(participant_set):
        add_anomaly(row_num, 'split_detail_participant_mismatch', 'review', 'split_details', f'Detail people {sorted(detail_people)} do not match split_with {sorted(participant_set)}.', 'Ask user which participant list is correct.')

    if row['amount_rounded'] is not None and split_details:
        values = [detail['value'] for detail in split_details if detail.get('value') is not None]
        if row['split_type'] == 'unequal' and values and sum(values) != row['amount_rounded']:
            add_anomaly(row_num, 'unequal_split_total_mismatch', 'error', 'split_details', f'Unequal split sums to {sum(values)}, amount is {row["amount_rounded"]}.', 'Hold row until split amounts sum to the expense amount.')
        if row['split_type'] == 'percentage' and values and sum(values) != Decimal('100'):
            add_anomaly(row_num, 'percentage_split_not_100', 'error', 'split_details', f'Percentages sum to {sum(values)}%, not 100%.', 'Hold row until percentages are corrected.')
        if row['split_type'] == 'share' and values and sum(values) <= 0:
            add_anomaly(row_num, 'share_split_zero_total', 'error', 'split_details', 'Share weights sum to zero or less.', 'Hold row until share weights are corrected.')

    if not pd.isna(parsed_date):
        for person in participant_set | ({row['paid_by_normalized']} if row['paid_by_normalized'] else set()):
            if person not in MEMBERSHIP_WINDOWS:
                continue
            start, end = MEMBERSHIP_WINDOWS[person]
            if parsed_date < start or (end is not None and parsed_date > end):
                add_anomaly(row_num, 'member_outside_active_window', 'review', 'membership', f'{person} appears on {parsed_date.date()}, outside active membership window.', 'Ask user whether this person should be charged/credited for the row.')

    matched_terms = [term for term in REVIEW_NOTE_TERMS if term in compact_text(f'{row["description_raw"]} {row["notes"]}')]
    if matched_terms:
        add_anomaly(row_num, 'source_note_review_signal', 'review', 'notes', f'Source text contains review signal(s): {matched_terms}.', 'Show note to user before final import.')


rows = normalized_expenses.to_dict('records')
for left_index in range(len(rows)):
    for right_index in range(left_index + 1, len(rows)):
        left = rows[left_index]
        right = rows[right_index]
        if pd.isna(left['parsed_date']) or pd.isna(right['parsed_date']):
            continue
        if left['parsed_date'] != right['parsed_date']:
            continue
        left_tokens = set(left['description_key'].split())
        right_tokens = set(right['description_key'].split())
        ordered_similarity = SequenceMatcher(None, left['description_key'], right['description_key']).ratio()
        sorted_similarity = SequenceMatcher(
            None,
            ' '.join(sorted(left_tokens)),
            ' '.join(sorted(right_tokens)),
        ).ratio()
        token_overlap = len(left_tokens & right_tokens) / len(left_tokens | right_tokens) if left_tokens | right_tokens else 0
        similarity = max(ordered_similarity, sorted_similarity, token_overlap)
        same_amount = left['amount_rounded'] == right['amount_rounded'] and left['currency_normalized'] == right['currency_normalized']
        same_payer = left['paid_by_normalized'] == right['paid_by_normalized']
        same_people = set(left['participants']) == set(right['participants'])
        overlapping_people = bool(set(left['participants']) & set(right['participants']))

        if similarity >= 0.72 and same_amount and same_payer and same_people:
            message = f'Looks like duplicate of row {right["raw_row_number"]}: description similarity {similarity:.0%}.'
            add_anomaly(left['raw_row_number'], 'probable_duplicate', 'review', 'description', message, 'Keep one row only after user approval.', True, str(right['raw_row_number']))
            message = f'Looks like duplicate of row {left["raw_row_number"]}: description similarity {similarity:.0%}.'
            add_anomaly(right['raw_row_number'], 'probable_duplicate', 'review', 'description', message, 'Keep one row only after user approval.', True, str(left['raw_row_number']))
        elif similarity >= 0.70 and overlapping_people and not (same_amount and same_payer):
            message = f'Possible duplicate/conflict with row {right["raw_row_number"]}: similar description but payer or amount differs.'
            add_anomaly(left['raw_row_number'], 'possible_conflicting_duplicate', 'review', 'description', message, 'Ask user which row is correct or whether both are valid.', True, str(right['raw_row_number']))
            message = f'Possible duplicate/conflict with row {left["raw_row_number"]}: similar description but payer or amount differs.'
            add_anomaly(right['raw_row_number'], 'possible_conflicting_duplicate', 'review', 'description', message, 'Ask user which row is correct or whether both are valid.', True, str(left['raw_row_number']))


anomaly_report = pd.DataFrame(anomalies)
if anomaly_report.empty:
    anomaly_report = pd.DataFrame(columns=['raw_row_number', 'code', 'severity', 'field', 'message', 'action', 'requires_user_approval', 'related_rows'])
else:
    anomaly_report = anomaly_report.sort_values(['raw_row_number', 'severity', 'code']).reset_index(drop=True)

print(f'Detected {len(anomaly_report)} anomaly records across {anomaly_report.raw_row_number.nunique()} CSV rows.')
display(anomaly_report.groupby(['severity', 'code']).size().reset_index(name='count').sort_values(['severity', 'code']))
display(anomaly_report)


In [ ]:
def build_split_rows(expense_row):
    if expense_row['amount_rounded'] is None or expense_row['currency_normalized'] not in FX_RATES_TO_INR:
        return []
    if not expense_row['split_type'] or expense_row['split_type'] not in supported_split_types:
        return []
    if not expense_row['participants']:
        return []

    split_type = expense_row['split_type']
    details = expense_row['split_details_parsed']
    fx_rate = expense_row['fx_rate_to_inr']

    people = []
    amounts = []
    split_basis = ''

    if split_type == 'equal':
        people = expense_row['participants']
        amounts = allocate_weighted_amount(expense_row['amount_rounded'], [1] * len(people))
        split_basis = 'equal'

    elif split_type == 'share':
        valid_details = [detail for detail in details if detail.get('value') is not None]
        if not valid_details or sum(detail['value'] for detail in valid_details) <= 0:
            return []
        people = [detail['person'] for detail in valid_details]
        amounts = allocate_weighted_amount(expense_row['amount_rounded'], [detail['value'] for detail in valid_details])
        split_basis = 'share weights'

    elif split_type == 'percentage':
        valid_details = [detail for detail in details if detail.get('value') is not None]
        if not valid_details or sum(detail['value'] for detail in valid_details) != Decimal('100'):
            return []
        people = [detail['person'] for detail in valid_details]
        amounts = allocate_weighted_amount(expense_row['amount_rounded'], [detail['value'] for detail in valid_details])
        split_basis = 'percentages'

    elif split_type == 'unequal':
        valid_details = [detail for detail in details if detail.get('value') is not None]
        if not valid_details:
            return []
        people = [detail['person'] for detail in valid_details]
        amounts = [money(detail['value']) for detail in valid_details]
        split_basis = 'explicit amounts'

    split_rows = []
    for person, share_amount in zip(people, amounts):
        split_rows.append({
            'raw_row_number': expense_row['raw_row_number'],
            'expense_date': expense_row['parsed_date'],
            'description': expense_row['description_raw'],
            'paid_by': expense_row['paid_by_normalized'],
            'participant': person,
            'currency': expense_row['currency_normalized'],
            'expense_amount_original_currency': expense_row['amount_rounded'],
            'share_amount_original_currency': share_amount,
            'share_amount_inr': (share_amount * fx_rate).quantize(CENTS, rounding=ROUND_HALF_UP),
            'split_type': split_type,
            'split_basis': split_basis,
        })
    return split_rows


split_records = []
for expense in normalized_expenses.to_dict('records'):
    split_records.extend(build_split_rows(expense))

splits_long = pd.DataFrame(split_records)
print(f'Built {len(splits_long)} split rows from parseable expense candidates.')
display(splits_long.head(20))


In [ ]:
approval_rows = set(anomaly_report.loc[anomaly_report['requires_user_approval'] == True, 'raw_row_number'])
settlement_rows = set(normalized_expenses.loc[normalized_expenses['is_settlement_candidate'], 'raw_row_number'])


def classify_import_status(row):
    if row['raw_row_number'] in settlement_rows:
        return 'payment_or_settlement_candidate'
    if row['raw_row_number'] in approval_rows:
        return 'needs_user_review'
    return 'ready_to_import'


normalized_expenses['import_status'] = normalized_expenses.apply(classify_import_status, axis=1)

if not splits_long.empty:
    splits_long = splits_long.merge(
        normalized_expenses[['raw_row_number', 'import_status']],
        on='raw_row_number',
        how='left',
    )

cleaned_expenses = normalized_expenses.loc[
    normalized_expenses['import_status'] == 'ready_to_import'
].copy()

review_queue = normalized_expenses.loc[
    normalized_expenses['import_status'] == 'needs_user_review'
].copy()

payments_or_settlements = normalized_expenses.loc[
    normalized_expenses['import_status'] == 'payment_or_settlement_candidate'
].copy()

ready_splits = splits_long.loc[
    splits_long['import_status'] == 'ready_to_import'
].copy() if not splits_long.empty else pd.DataFrame()

people = sorted({
    person
    for people_list in normalized_expenses['participants']
    for person in people_list
} | set(normalized_expenses['paid_by_normalized'].dropna()) - {''})

paid_by_person = {person: Decimal('0.00') for person in people}
for expense in cleaned_expenses.to_dict('records'):
    payer = expense['paid_by_normalized']
    if payer:
        paid_by_person[payer] = (paid_by_person.get(payer, Decimal('0.00')) + expense['amount_inr']).quantize(CENTS)

owed_by_person = {person: Decimal('0.00') for person in people}
for split in ready_splits.to_dict('records'):
    participant = split['participant']
    owed_by_person[participant] = (owed_by_person.get(participant, Decimal('0.00')) + split['share_amount_inr']).quantize(CENTS)

balance_summary = pd.DataFrame([
    {
        'person': person,
        'paid_inr_ready_rows': paid_by_person.get(person, Decimal('0.00')),
        'owed_inr_ready_rows': owed_by_person.get(person, Decimal('0.00')),
        'net_balance_inr_ready_rows': (paid_by_person.get(person, Decimal('0.00')) - owed_by_person.get(person, Decimal('0.00'))).quantize(CENTS),
    }
    for person in people
]).sort_values('person').reset_index(drop=True)

print('Import status counts:')
display(normalized_expenses['import_status'].value_counts().rename_axis('status').reset_index(name='rows'))

print('Rows ready to import without user approval:', len(cleaned_expenses))
print('Rows requiring review:', len(review_queue))
print('Payment/settlement candidates:', len(payments_or_settlements))

print('\nCleaned expense candidates:')
display(cleaned_expenses[[
    'raw_row_number', 'parsed_date', 'description_raw', 'paid_by_normalized',
    'amount_rounded', 'currency_normalized', 'amount_inr', 'split_type', 'participants_text'
]])

print('\nReview queue:')
display(review_queue[[
    'raw_row_number', 'date_raw', 'description_raw', 'paid_by_raw',
    'amount_raw', 'currency_raw', 'split_type_raw', 'split_with_raw', 'notes'
]])

print('\nPayment/settlement candidates:')
display(payments_or_settlements[[
    'raw_row_number', 'date_raw', 'description_raw', 'paid_by_raw',
    'amount_raw', 'currency_raw', 'split_type_raw', 'split_with_raw', 'notes'
]])

print('\nBalance preview using only ready-to-import rows:')
display(balance_summary)


## How This Maps To The App Later

The app importer can reuse this shape:

- `raw_expenses`: store/upload audit trail.
- `normalized_expenses`: candidate expense records after safe normalization.
- `splits_long`: per-person obligations for balance tracing.
- `anomaly_report`: user-facing import report.
- `review_queue`: rows the app must ask the user to approve or correct.
- `payments_or_settlements`: rows that should go through a payment/settlement flow, not the expense flow.

This prototype intentionally blocks questionable rows from `cleaned_expenses` so the final app does not make silent guesses.
